# Лабораторная работа

1. Генерация выборок из **N(0, 1)** и **U[0, 1]**
2. Гистограмма и ядерная оценка плотности (гауссово ядро) для n = 10, 100, 1000, 10000
3. Оценки квантилей 1%, 5%, 50% тремя способами
4. Повтор пунктов 2–3 не менее N = 1000 раз: оценка дисперсии квантилей и выводы


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import ndtr

np.random.seed(42)

SIZES = [10, 100, 1000, 10000]
LEVELS = [0.01, 0.05, 0.50]
N_MC = 1000  # число повторов для оценки дисперсии


## 1. Генерация выборок


In [ ]:
def gen_normal(n):
    return np.random.normal(0, 1, n)

def gen_uniform(n):
    return np.random.uniform(0, 1, n)

print('N(0,1):', gen_normal(5))
print('U[0,1]:', gen_uniform(5))


## 2. Гистограмма и ядерная оценка плотности

Число полос гистограммы:

$$
k \approx 1 + 1{,}59 \cdot \ln n
$$

ЯОП с гауссовым ядром, ширина окна по Сильверману: $h = 1{,}06\,\hat\sigma\, n^{-1/5}$.


In [ ]:
def n_bins(n):
    return max(1, round(1 + 1.59 * math.log(n)))

def make_hist(sample):
    k = n_bins(len(sample))
    counts, edges = np.histogram(sample, bins=k)
    width = edges[1] - edges[0]
    density = counts / (len(sample) * width)
    return edges, density

def bandwidth(sample):
    n = len(sample)
    sigma = np.std(sample, ddof=1) if n > 1 else 1.0
    if sigma <= 0:
        sigma = 1.0
    return 1.06 * sigma * n ** (-0.2)

def make_kde(sample, grid):
    """ЯОП на сетке (для графиков)."""
    n = len(sample)
    h = bandwidth(sample)
    dens = np.zeros_like(grid, dtype=float)
    # считаем порциями, чтобы не раздувать память на больших n
    step = 500
    for i in range(0, n, step):
        part = sample[i:i + step]
        z = (grid[:, None] - part[None, :]) / h
        dens += np.exp(-0.5 * z * z).sum(axis=1)
    dens /= (n * h * np.sqrt(2 * np.pi))
    return dens, h

for n in SIZES:
    print(f'n = {n:5d}  →  k = {n_bins(n)}')


### Графики для N(0, 1)


In [ ]:
fig, axes = plt.subplots(len(SIZES), 2, figsize=(10, 12))

for i, n in enumerate(SIZES):
    sample = gen_normal(n)
    edges, dens_h = make_hist(sample)

    ax = axes[i, 0]
    ax.bar(edges[:-1], dens_h, width=np.diff(edges), align='edge',
           alpha=0.6, edgecolor='black', label='гистограмма')
    x = np.linspace(edges[0], edges[-1], 300)
    ax.plot(x, stats.norm.pdf(x), 'r-', label='истинная плотность')
    ax.set_title(f'Гистограмма, n={n}, k={n_bins(n)}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    ax = axes[i, 1]
    grid = np.linspace(sample.min() - 1, sample.max() + 1, 400)
    dens_k, h = make_kde(sample, grid)
    ax.plot(grid, dens_k, 'b-', label=f'ЯОП, h={h:.3f}')
    ax.plot(grid, stats.norm.pdf(grid), 'r-', label='истинная плотность')
    ax.set_title(f'ЯОП, n={n}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('N(0, 1)', fontsize=14)
plt.tight_layout()
plt.show()


### Графики для U[0, 1]


In [ ]:
fig, axes = plt.subplots(len(SIZES), 2, figsize=(10, 12))

for i, n in enumerate(SIZES):
    sample = gen_uniform(n)
    edges, dens_h = make_hist(sample)

    ax = axes[i, 0]
    ax.bar(edges[:-1], dens_h, width=np.diff(edges), align='edge',
           alpha=0.6, edgecolor='black', label='гистограмма')
    x = np.linspace(-0.1, 1.1, 300)
    ax.plot(x, stats.uniform.pdf(x), 'r-', label='истинная плотность')
    ax.set_title(f'Гистограмма, n={n}, k={n_bins(n)}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    ax = axes[i, 1]
    grid = np.linspace(-0.3, 1.3, 400)
    dens_k, h = make_kde(sample, grid)
    ax.plot(grid, dens_k, 'b-', label=f'ЯОП, h={h:.3f}')
    ax.plot(grid, stats.uniform.pdf(grid), 'r-', label='истинная плотность')
    ax.set_title(f'ЯОП, n={n}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('U[0, 1]', fontsize=14)
plt.tight_layout()
plt.show()


## 3. Оценки квантилей

1. **По выборочной ФР** — $q_p = X_{(\lceil np \rceil)}$.
2. **По гистограмме** — по полосам строим ФР (внутри полосы масса равномерна) и обращаем её.
3. **По ЯОП** — для гауссова ядра
   $\hat F(x)=\frac{1}{n}\sum_{i=1}^n \Phi\left(\frac{x-X_i}{h}\right)$;
   квантиль находим обращением $\hat F$ (это ФР именно ядерной оценки, а не выборочная ФР).


In [ ]:
def quantile_empirical(sample, p):
    xs = np.sort(sample)
    idx = math.ceil(len(sample) * p) - 1
    idx = min(max(idx, 0), len(xs) - 1)
    return float(xs[idx])


def quantile_from_hist(edges, density, p):
    """Квантиль только по гистограмме (равномерно внутри полосы)."""
    widths = np.diff(edges)
    mass = density * widths
    cum = np.concatenate([[0.0], np.cumsum(mass)])
    j = int(np.searchsorted(cum, p, side='left') - 1)
    j = min(max(j, 0), len(mass) - 1)
    if mass[j] <= 0:
        return float(edges[j])
    frac = (p - cum[j]) / mass[j]
    frac = min(max(frac, 0.0), 1.0)
    return float(edges[j] + frac * widths[j])


def quantiles_from_kde(sample, levels, h=None):
    """
    Квантили по ЯОП: решаем F̂(x)=p, где
    F̂(x) = (1/n) Σ Φ((x - X_i)/h) — ФР гауссовой ядерной оценки.
    """
    if h is None:
        h = bandwidth(sample)
    lo = float(np.min(sample)) - 5 * h
    hi = float(np.max(sample)) + 5 * h
    out = np.empty(len(levels))
    for i, p in enumerate(levels):
        a, b = lo, hi
        for _ in range(40):
            mid = 0.5 * (a + b)
            if ndtr((mid - sample) / h).mean() < p:
                a = mid
            else:
                b = mid
        out[i] = b
    return out


def three_quantiles(sample):
    """Три оценки квантилей для всех LEVELS."""
    edges, dens_h = make_hist(sample)
    h = bandwidth(sample)
    q_kde = quantiles_from_kde(sample, LEVELS, h)

    q_emp = np.empty(len(LEVELS))
    q_hist = np.empty(len(LEVELS))
    for i, p in enumerate(LEVELS):
        q_emp[i] = quantile_empirical(sample, p)
        q_hist[i] = quantile_from_hist(edges, dens_h, p)
    return q_emp, q_hist, q_kde


In [ ]:
def print_example_table(law_name, sampler):
    print(f'\n===== {law_name} (один прогон) =====')
    print(f'{"n":>6} {"p":>6} {"истина":>10} {"ФР":>10} {"гистогр.":>10} {"ЯОП":>10}')
    for n in SIZES:
        sample = sampler(n)
        q_emp, q_hist, q_kde = three_quantiles(sample)
        for i, p in enumerate(LEVELS):
            q_true = stats.norm.ppf(p) if law_name == 'normal' else stats.uniform.ppf(p)
            print(f'{n:6d} {p:6.2f} {q_true:10.4f} {q_emp[i]:10.4f} {q_hist[i]:10.4f} {q_kde[i]:10.4f}')

np.random.seed(42)
print_example_table('normal', gen_normal)
print_example_table('uniform', gen_uniform)


## 4. Оценка дисперсии квантилей (N = 1000 повторов)

Для каждого закона и каждого $n$ повторяем генерацию выборки и расчёт трёх оценок квантилей $N = 1000$ раз.
По $N$ значениям оцениваем выборочную дисперсию:

$$
\widehat{\mathrm{Var}}(\hat q_p)
= \frac{1}{N-1}\sum_{j=1}^{N}\bigl(\hat q_p^{(j)} - \overline{\hat q_p}\bigr)^2
$$

Чем меньше дисперсия, тем стабильнее (информативнее) оценка.

> Ячейка ниже может считаться несколько минут.


In [ ]:
def mc_variances(law_name, n, n_mc=N_MC, seed=0):
    rng = np.random.default_rng(seed)
    emp = np.empty((n_mc, len(LEVELS)))
    hist = np.empty((n_mc, len(LEVELS)))
    kde = np.empty((n_mc, len(LEVELS)))

    for j in range(n_mc):
        if law_name == 'normal':
            sample = rng.normal(0.0, 1.0, n)
        else:
            sample = rng.uniform(0.0, 1.0, n)

        q_emp, q_hist, q_kde = three_quantiles(sample)
        emp[j] = q_emp
        hist[j] = q_hist
        kde[j] = q_kde

        if (j + 1) % 250 == 0:
            print(f'  {law_name}, n={n}: {j+1}/{n_mc}')

    return emp.var(axis=0, ddof=1), hist.var(axis=0, ddof=1), kde.var(axis=0, ddof=1)


results = {}
for law_name in ['normal', 'uniform']:
    print(f'\n===== MC: {law_name} =====')
    for n in SIZES:
        print(f'считаем n = {n} ...')
        results[(law_name, n)] = mc_variances(law_name, n, n_mc=N_MC, seed=1000 + n + (0 if law_name == 'normal' else 50000))
print('\nготово')


In [ ]:
for law_name in ['normal', 'uniform']:
    print(f'\n===== Дисперсии: {law_name}, N_MC={N_MC} =====')
    print(f'{"n":>6} {"p":>6} {"Var ФР":>12} {"Var гист.":>12} {"Var ЯОП":>12}')
    for n in SIZES:
        v_emp, v_hist, v_kde = results[(law_name, n)]
        for i, p in enumerate(LEVELS):
            print(f'{n:6d} {p:6.2f} {v_emp[i]:12.6f} {v_hist[i]:12.6f} {v_kde[i]:12.6f}')


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True)

for row, law_name in enumerate(['normal', 'uniform']):
    for col, p in enumerate(LEVELS):
        ax = axes[row, col]
        ve, vh, vk = [], [], []
        for n in SIZES:
            v_emp, v_hist, v_kde = results[(law_name, n)]
            ve.append(v_emp[col])
            vh.append(v_hist[col])
            vk.append(v_kde[col])
        ax.plot(SIZES, ve, 'o-', label='ФР')
        ax.plot(SIZES, vh, 's-', label='гистограмма')
        ax.plot(SIZES, vk, '^-', label='ЯОП')
        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_title(f'{law_name}, p={p}')
        ax.set_xlabel('n')
        ax.set_ylabel('дисперсия')
        ax.grid(True, which='both', alpha=0.3)
        if row == 0 and col == 0:
            ax.legend(fontsize=8)

plt.suptitle(f'Оценка дисперсии квантилей (N_MC = {N_MC})', fontsize=13)
plt.tight_layout()
plt.show()


### Выводы об информативности оценок

- **Выборочная ФР** обычно даёт **наименьшую дисперсию**: квантиль берётся напрямую из порядковых статистик, без огрубления и лишнего сглаживания.
- **Гистограмма** — самая грубая: мало полос и равномерность внутри полосы увеличивают разброс, особенно для **1%** и **5%** квантилей при малых $n$.
- **ЯОП** часто близка к ФР на **медиане**, но на хвостах и у **U[0, 1]** (вытекание ядра за \([0,1]\)) может быть менее информативной.
- С ростом $n$ дисперсия **всех** методов уменьшается.

**Итог:** наиболее информативна оценка по **выборочной ФР**; дальше обычно **ЯОП**; наименее информативна **гистограмма**.
